# Carcaj de Brauer $Q_{\mathfrak{m}}$ — Red de citación de 17 artículosConfiguración de Brauer $\Gamma=(M,\mathfrak{m},\mu,O)$:- $M$ = **621** autores citados (vértices)- $\mathfrak{m}=\{e_1,\dots,e_{17}\}$ = los 17 artículos (multiconjuntos)- $\mu(v)=1$ si $\mathrm{val}(v)>1$, y $\mu(v)=2$ si $\mathrm{val}(v)=1$  *(multiplicidad de Tipo M)*- $O$ = orden cronológico: $e_i<e_j \iff i<j$**Sube `matriz_occ_V2.csv` cuando se te pida.**

## 1. Preparación

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport collectionstry:                       # Colab    from google.colab import files    up = files.upload()    CSV = list(up.keys())[0]except Exception:          # local    CSV = 'matriz_occ_V2.csv'df = pd.read_csv(CSV)print(df.shape)df.head()

## 2. La matriz de ocurrencias $\Omega$$\Omega_{ij}=\mathrm{occ}(v_i,e_j)$ = referencias del artículo $e_j$ en las que aparece $v_i$.

In [ ]:
N = 17COLS = [f'e_{j}' for j in range(1, N+1)]Omega   = df[COLS].to_numpy(dtype=int)    # (|M| x 17)autores = df['autor'].tolist()etiq    = df['v_i'].tolist()val = Omega.sum(axis=1)                   # val(v_i)  -> suma de filamu  = np.where(val > 1, 1, 2)             # multiplicidad Tipo Mtam = Omega.sum(axis=0)                   # |e_j|     -> suma de columnaprint(f"|M| = {len(val)}   |m| = {N}   sum val = {val.sum()}")print(f"mu=1: {(mu==1).sum()} vertices    mu=2: {(mu==2).sum()} vertices")print("\nTamanos |e_j|:", dict(zip(COLS, tam)))

## 3. Parámetros`VAL_MIN`: solo se dibujan los vértices con $\mathrm{val}(v)\geq$ `VAL_MIN`.| VAL_MIN | vértices | flechas | bucles ||---|---|---|---|| 1 | 621 | 1083 | 750 || 2 | 122 | 584 | 251 || 3 | 65 | 470 | 183 || 4 | 48 | 419 | 152 || 5 | 28 | 339 | 128 |`MODO`: `'agregado'` funde las flechas paralelas (grosor = cantidad) y resume los bucles;`'completo'` dibuja cada $\alpha^v_k$ con un color por vértice.

In [ ]:
VAL_MIN = 3          # <--- cambia este valorMODO    = 'agregado' # 'agregado'  o  'completo'

## 4. Secuencias de sucesores y construcción de $Q_{\mathfrak{m}}$$S_v$ lista los multiconjuntos que contienen a $v$, repetidos según $\mathrm{occ}(v,e_j)$ yen el orden de $O$. Se lee **cíclicamente**: cada par consecutivo $(e_j,e_{j'})$ aporta unaflecha $\alpha^v_k : e_j \to e_{j'}$.

In [ ]:
def sucesores(i):    # S_{v_i}: lista ciclica de indices de multiconjuntos (1..17)    S = []    for j in range(N):        S += [j+1] * Omega[i, j]    return Sdef flechas(i):    # Flechas que v_i aporta a Q_1: (origen, destino, indice_v, k)    S = sucesores(i)    L = len(S)    return [(S[k], S[(k+1) % L], i, k+1) for k in range(L)]activos = [i for i in range(len(val)) if val[i] >= VAL_MIN]Q1 = [a for i in activos for a in flechas(i)]nbucles = sum(1 for u, v, _, _ in Q1 if u == v)print(f"VAL_MIN = {VAL_MIN}")print(f"vertices de M activos : {len(activos)}")print(f"|Q_1| (flechas)       : {len(Q1)}   de las cuales {nbucles} son bucles")print(f"|Q_0|                 : {N}")print(f"\nEjemplo  S_{{{etiq[0]}}}  ({autores[0]}), longitud {val[0]}:")print(sucesores(0))

## 5. Dibujo del carcaj

In [ ]:
def dibujar(Q1, activos, val_min, modo='agregado', figsize=12, guardar=None):    pos = {j: (np.cos(np.pi/2 - 2*np.pi*(j-1)/N),               np.sin(np.pi/2 - 2*np.pi*(j-1)/N)) for j in range(1, N+1)}    cmap = plt.get_cmap('tab20')    color = {v: cmap(n % 20) for n, v in enumerate(activos)}    bucles, pares = collections.defaultdict(list), collections.defaultdict(list)    for u, v, i, k in Q1:        (bucles[u] if u == v else pares[(u, v)]).append(i)    fig, ax = plt.subplots(figsize=(figsize, figsize))    if modo == 'agregado':        for (u, v), lst in pares.items():            ax.annotate("", xy=pos[v], xytext=pos[u],                arrowprops=dict(arrowstyle="-|>", color='#37474F',                                lw=0.4 + 0.28*len(lst), alpha=.55,                                shrinkA=17, shrinkB=17,                                connectionstyle="arc3,rad=0.15"))        for j, lst in bucles.items():            d = np.array(pos[j]); d = d/np.linalg.norm(d)            r = 0.045 + 0.012*np.sqrt(len(lst))            ax.add_patch(plt.Circle(tuple(d*(1+r*1.7)), r, fill=False,                                    ec='#B71C1C', lw=1.2 + 0.1*len(lst), alpha=.8))            ax.text(*(d*(1+r*3.4)), str(len(lst)), ha='center', va='center',                    fontsize=9, color='#B71C1C')    else:        for (u, v), lst in pares.items():            for n, i in enumerate(lst):                rad = 0.10*(n - (len(lst)-1)/2)                ax.annotate("", xy=pos[v], xytext=pos[u],                    arrowprops=dict(arrowstyle="-|>", color=color[i], lw=0.8, alpha=.7,                                    shrinkA=16, shrinkB=16,                                    connectionstyle=f"arc3,rad={rad}"))        for j, lst in bucles.items():            d = np.array(pos[j]); base = np.arctan2(d[1], d[0]); L = len(lst)            for n, i in enumerate(lst):                ang = base + (n - (L-1)/2) * (1.5/max(L, 1))                c = d + np.array([np.cos(ang), np.sin(ang)])*0.085                ax.add_patch(plt.Circle(tuple(c), 0.032, fill=False,                                        ec=color[i], lw=0.9, alpha=.75))    for j in range(1, N+1):        ax.scatter(*pos[j], s=950, c='white', ec='black', zorder=5, lw=1.6)        ax.text(*pos[j], f"$e_{{{j}}}$", ha='center', va='center', zorder=6, fontsize=11)    nb = sum(len(v) for v in bucles.values())    ax.set_aspect('equal'); ax.axis('off'); ax.margins(.22)    ax.set_title(f"Carcaj de Brauer $Q_\\mathfrak{{m}}$  —  val$(v)\\geq{val_min}$\n"                 f"{len(activos)} vertices de $M$  ·  {len(Q1)} flechas ({nb} bucles)  ·  modo: {modo}",                 fontsize=13)    plt.tight_layout()    if guardar: plt.savefig(guardar, dpi=200, bbox_inches='tight')    plt.show()dibujar(Q1, activos, VAL_MIN, MODO, guardar=f'carcaj_val{VAL_MIN}_{MODO}.png')

## 6. Sub-carcaj de un solo vérticeAísla el ciclo especial $C_v$ de un autor. Ahora con `guardar=` para exportar el PNG.**Ojo con el índice:** las filas son 0-based y las etiquetas $v_i$ son 1-based, así que$v_1$ (Cañadas) es `subcarcaj(0)`. Usa `por_nombre()` para no llevar la cuenta.

In [ ]:
def por_nombre(txt):    # Devuelve el indice de fila del primer autor que contenga txt    for k, a in enumerate(autores):        if txt.lower() in a.lower():            return k    raise ValueError(f"no encontre '{txt}'")def subcarcaj(i, figsize=8, guardar=None):    S = sucesores(i)    pos = {j: (np.cos(np.pi/2 - 2*np.pi*(j-1)/N),               np.sin(np.pi/2 - 2*np.pi*(j-1)/N)) for j in range(1, N+1)}    fig, ax = plt.subplots(figsize=(figsize, figsize))    cont = collections.Counter()    for k in range(len(S)):        u, v = S[k], S[(k+1) % len(S)]        if u == v:            d = np.array(pos[u]); base = np.arctan2(d[1], d[0])            n = cont[u]; cont[u] += 1            ang = base + (n - 1)*0.45            c = d + np.array([np.cos(ang), np.sin(ang)])*0.085            ax.add_patch(plt.Circle(tuple(c), 0.035, fill=False, ec='#C62828', lw=1.4))        else:            n = cont[(u, v)]; cont[(u, v)] += 1            ax.annotate("", xy=pos[v], xytext=pos[u],                arrowprops=dict(arrowstyle="-|>", color='#1565C0', lw=1.3, alpha=.8,                                shrinkA=16, shrinkB=16,                                connectionstyle=f"arc3,rad={0.14 + 0.09*n}"))    for j in range(1, N+1):        act = j in S        ax.scatter(*pos[j], s=900, c='#FFF9C4' if act else 'white',                   ec='black', zorder=5, lw=1.5)        ax.text(*pos[j], f"$e_{{{j}}}$", ha='center', va='center', zorder=6, fontsize=10)    nb = sum(1 for k in range(len(S)) if S[k] == S[(k+1) % len(S)])    ax.set_aspect('equal'); ax.axis('off'); ax.margins(.2)    ax.set_title(f"Ciclo especial de {etiq[i]} = {autores[i]}\n"                 f"val = {val[i]},  mu = {mu[i]},  {(Omega[i]>0).sum()} articulos,  {nb} bucles",                 fontsize=12)    plt.tight_layout()    if guardar: plt.savefig(guardar, dpi=200, bbox_inches='tight')    plt.show()# los cinco de mayor valenciafor k in range(5):    subcarcaj(k, guardar=f'ciclo_v{k+1}.png')

## 7. Tabla resumen y exportación de $Q_1$

In [ ]:
resumen = pd.DataFrame({    'v_i': [etiq[i] for i in activos],    'autor': [autores[i] for i in activos],    'val': [val[i] for i in activos],    'mu': [mu[i] for i in activos],    'n_articulos': [(Omega[i] > 0).sum() for i in activos],    'n_flechas': [len(sucesores(i)) for i in activos],    'n_bucles': [sum(1 for u, v, _, _ in flechas(i) if u == v) for i in activos],})display(resumen.head(25))aristas = pd.DataFrame(    [{'origen': f'e_{u}', 'destino': f'e_{v}', 'v_i': etiq[i],      'autor': autores[i], 'k': k, 'bucle': u == v} for u, v, i, k in Q1])aristas.to_csv(f'Q1_val{VAL_MIN}.csv', index=False)print(f"\nQ1_val{VAL_MIN}.csv  ->  {len(aristas)} flechas")aristas.head(10)

## 8. Invariantes: relaciones, dimensión y entropía

In [ ]:
n1 = int((val == 1).sum())TI   = int(tam.sum() - N)                                  # sum (|e_j| - 1)TII  = int(val.sum())                                      # un ciclo especial por ocurrenciaTIII = int((tam**2).sum() - val[val >= 2].sum() - n1)      # pares no-subcamino, menos excepciondim = 2*N + int((val*(val*mu - 1)).sum())w = val*mu; delta = int(w.sum())def H(x):    p = np.asarray(x, float); p = p[p > 0]; p = p/p.sum()    return -(p*np.log2(p)).sum()print(f"Tipo I   = {TI}")print(f"Tipo II  = {TII}")print(f"Tipo III = {TIII}")print(f"TOTAL generadores de I_Gamma = {TI+TII+TIII}\n")print(f"dim Lambda = 2({N}) + {int((val*(val*mu-1)).sum())} = {dim}\n")print(f"delta = {delta}")print(f"H(Lambda) con val*mu = {H(w):.4f} bits   H/Hmax = {H(w)/np.log2(len(val)):.4f}   2^H = {2**H(w):.1f}")print(f"H         con val    = {H(val):.4f} bits   H/Hmax = {H(val)/np.log2(len(val)):.4f}   2^H = {2**H(val):.1f}")print(f"Hmax = log2({len(val)}) = {np.log2(len(val)):.4f}")